# Tutorial 5 — Bimodal-histogram analysis

**Goal:** go from two reconstructed volumes to material clusters, a segmentation, and
quantitative quality metrics.

**You will learn:** `compute_bimodal_histogram`, `fit_gmm` / `auto_fit_gmm`, `segment_by_gmm`,
`detect_artifact_signatures`, `evaluate_histogram_quality`, `compute_histogram_metrics` and
`compute_histogram_metrics_morphology_aware`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

In [ ]:
sim = nxs.DualModalitySimulation(preset="composite", N=48, n_angles=90, verbose=False)
clean = sim.run(tag="clean")
noisy = sim.run(nxs.ArtifactConfig.noise_only(I0=5e3), tag="noisy", ref_result=clean)
phantom = sim.phantom

## 1. Computing the histogram

`compute_bimodal_histogram(vol_x, vol_n)` bins every voxel pair. Pass a `mask` to restrict it,
e.g. to exclude the air around the sample, which otherwise dominates the counts.

In [ ]:
hist_all = nxs.compute_bimodal_histogram(noisy.vol_xray, noisy.vol_neutron, bins=128)
hist_obj = nxs.compute_bimodal_histogram(noisy.vol_xray, noisy.vol_neutron, bins=128,
                                         mask=phantom.label_vol > 0)
print("voxels:", hist_all.total_voxels, "→", hist_obj.total_voxels, "inside the sample")
fig = nxs.plot_bimodal_histogram(hist_obj, title="noisy run, sample voxels only")

## 2. Gaussian mixture model

Each material should form a roughly Gaussian blob, so a Gaussian mixture model (GMM) is a
natural way to find clusters **without** ground truth. `auto_fit_gmm` chooses the number of
components by the Bayesian information criterion (BIC).

In [ ]:
gmm = nxs.auto_fit_gmm(hist_obj, min_k=3, max_k=6)
print("chosen number of components:", gmm.n_components)
fig = nxs.plot_bimodal_histogram(hist_obj, gmm=gmm, title="GMM fit (2σ ellipses)")

## 3. Segmentation

Assigning each voxel to its most likely component gives a 3-D segmentation — the practical
payoff of dual-modality imaging.

In [ ]:
seg = nxs.segment_by_gmm(noisy.vol_xray, noisy.vol_neutron, gmm,
                         mask=phantom.label_vol > 0)
s = 24
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(phantom.label_vol[s], cmap="tab10", vmin=0, vmax=9); axes[0].set_title("ground truth")
axes[1].imshow(np.ma.masked_less(seg[s], 0), cmap="tab10", vmin=0, vmax=9); axes[1].set_title("GMM segmentation")
axes[2].imshow(noisy.vol_neutron[s], cmap="gray"); axes[2].set_title("neutron reconstruction")
for ax in axes: ax.axis("off")

Clusters are numbered arbitrarily; the metrics below match them to materials.

## 4. Artifact signatures (no ground truth needed)

Some artifacts leave characteristic shapes in the histogram. `detect_artifact_signatures`
turns them into scores; with a clean reference run it also measures the neutron shift.

In [ ]:
sig = nxs.detect_artifact_signatures(noisy.histogram, ref_hist=clean.histogram)
for k, v in vars(sig).items():
    if not isinstance(v, dict):
        print(f"{k:>26}: {v:.4f}")

## 5. Ground-truth-anchored quality metrics

In a simulation we know the exact material positions, so we can score the histogram:

* **ε_k / CE** — centroid error per material / mean (cm⁻¹)
* **σ_x, σ_n** — cluster spread along each axis
* **DB** — Davies–Bouldin separability index
* **O_ab** — fraction of voxels misassigned between neighbouring materials

There are three flavours, from quick to thorough:

In [ ]:
m = nxs.evaluate_histogram_quality(hist_obj, phantom, gmm_n_init=2)
print(m.summary())

`compute_histogram_metrics` adds shape metrics, pathology warnings and table exports
(`to_dataframe()`, `to_csv()`). `gt_seeded_gmm=True` seeds the mixture at the true positions
so rare phases are not swallowed by abundant ones.

In [ ]:
table = nxs.compute_histogram_metrics(phantom, hist_obj, gt_seeded_gmm=True,
                                      ref_hist=clean.histogram)
print(table)
table.to_dataframe().head()

The **morphology-aware** metrics skip clustering entirely: they ask "where do the voxels
that truly are iron end up?" using the (eroded) ground-truth label mask. They cannot be fooled
by a mixture component placed in the wrong spot, and in `morphology_explore` mode they report
every connected region separately.

In [ ]:
table_la, regions = nxs.compute_histogram_metrics_morphology_aware(
    phantom, hist_all, noisy.vol_xray, noisy.vol_neutron, mode="morphology_explore")
print("CE =", round(table_la.scalars["CE"], 4), " DB =", round(table_la.scalars["DB"], 4))
for r in regions[:6]:
    print(f"{r.material:>6} region {r.region_id}: {r.n_voxels_total:5d} voxels, ε = {r.eps_k:.3f}")

**Exercise:** compare `evaluate_histogram_quality` and the morphology-aware CE for the
realistic artifact configuration. Why can the GMM-based CE look *better* than the label-anchored one?